# Creating Train and Test Sets with Explicit Feedback
- Dataset: MovieLens32m
- Split Method: leave-last-out/leave-one-out


source: https://github.com/yihong-chen/neural-collaborative-filtering/blob/master/src/data.py

In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

np.random.seed(42)

In [2]:
def set_dtypes(dataset: pd.DataFrame) -> pd.DataFrame:
    dataset['userId'] = dataset['userId'].astype(np.uint32)
    dataset['rating'] = dataset['rating'].astype(np.float16)
    dataset['movieId'] = dataset['movieId'].astype(np.uint32)
    dataset['timestamp'] = dataset['timestamp'].astype(np.uint32)

    return dataset

In [ ]:
ratings = pd.read_csv('../../data/ml-32m/ratings.csv')
ratings = set_dtypes(ratings)

In [4]:
user_pool = set(ratings['userId'].unique())
item_pool = set(ratings['movieId'].unique())

### Remap IDs to be contiguous

In [5]:
user_id_map = {uid: idx for idx, uid in enumerate(ratings['userId'].unique())}
item_id_map = {mid: idx for idx, mid in enumerate(ratings['movieId'].unique())}

ratings['userId'] = ratings['userId'].map(user_id_map).astype(np.int64)
ratings['movieId'] = ratings['movieId'].map(item_id_map).astype(np.int64)

### Pop last rated item

In [6]:
def pop_last_user_review(ratings_df):
    # Sort by timestamp to ensure the last review is at the end
    ratings_df = ratings_df.sort_values(by='timestamp')
    
    # Get the last review for each user
    last_reviews = ratings_df.groupby('userId').tail(1)
    
    # Remove the last reviews from the original DataFrame
    ratings_without_last = ratings_df[~ratings_df.index.isin(last_reviews.index)]
    
    return ratings_without_last, last_reviews

In [7]:
ratings_without_last, last_ratings = pop_last_user_review(ratings)

### Export as train and test sets

In [8]:
ratings_without_last = ratings_without_last.drop(columns=['timestamp'])

In [9]:
last_ratings = last_ratings.drop(columns=['timestamp'])

In [ ]:
import os

os.makedirs('../../data/explicit', exist_ok=True)
ratings_without_last.to_csv("../../data/explicit/train.csv", index=False)
last_ratings.to_csv("../../data/explicit/test.csv", index=False)

# Create smaller subsets of data

In [ ]:
import pandas as pd
import numpy as np
import os

os.makedirs('../../data/explicit', exist_ok=True)

train = pd.read_csv('../../data/explicit/train.csv')
test = pd.read_csv('../../data/explicit/test.csv')

# Sample ~10% of users to keep train/test alignment
np.random.seed(42)
all_users = test['userId'].unique()
subset_users = np.random.choice(all_users, size=int(len(all_users) * 0.1), replace=False)

train_subset = train[train['userId'].isin(subset_users)]
test_subset = test[test['userId'].isin(subset_users)]

train_subset.to_csv('../../data/explicit/train_subset.csv', index=False)
test_subset.to_csv('../../data/explicit/test_subset.csv', index=False)

print(f'Train subset: {len(train_subset)} rows, {train_subset.userId.nunique()} users')
print(f'Test subset: {len(test_subset)} rows, {test_subset.userId.nunique()} users')

Train subset: 3262062 rows, 20094 users
Test subset: 20094 rows, 20094 users
